# Chzzk Video Transcriber (Google Colab)

치지직(Chzzk) 다시보기 영상의 특정 구간을 다운로드하고, AI 음성인식(faster-whisper)으로 트랜스크립트를 생성합니다.

**주요 기능:**
- 치지직 VOD 구간 다운로드
- faster-whisper 기반 한국어 음성인식 (GPU 가속)
- 화자분리 (WeSpeaker / Simple Diarizer - HuggingFace 토큰 불필요)
- 채팅 수집 및 트랜스크립트 동기화
- SRT 자막 파일 생성

**사용법:** 위에서부터 순서대로 셀을 실행하세요. GPU 런타임을 사용하면 훨씬 빠릅니다.

> **GPU 설정:** 런타임 → 런타임 유형 변경 → T4 GPU 선택

## 1. 패키지 설치
필요한 패키지를 설치합니다. (최초 1회, 약 2-3분 소요)

In [ ]:
# 패키지 설치
!pip install -q faster-whisper>=1.1.0
!pip install -q simple-diarizer
!pip install -q ffmpeg-python
!pip install -q requests tqdm

# WeSpeaker 설치 (선택사항 - 실패해도 simple-diarizer로 대체)
!pip install -q "git+https://github.com/wenet-e2e/wespeaker.git" 2>/dev/null || echo "WeSpeaker 설치 실패 - Simple Diarizer를 사용합니다."

# FFmpeg 시스템 패키지 확인
!which ffmpeg || (apt-get update -qq && apt-get install -y -qq ffmpeg)

print("\n=== 설치 완료 ===")

# GPU 확인
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("GPU 없음 - CPU 모드로 실행됩니다 (속도가 느릴 수 있음)")

## 2. 설정
영상 URL, 구간, 모델 등을 설정합니다.

In [ ]:
#@title 설정 {display-mode: "form"}

#@markdown ### 영상 정보
VIDEO_URL = "https://chzzk.naver.com/video/12345" #@param {type:"string"}
START_TIME = "00:00:00" #@param {type:"string"}
END_TIME = "00:05:00" #@param {type:"string"}

#@markdown ### Whisper 모델
#@markdown - `large-v3`: 최고 정확도 (VRAM ~10GB)
#@markdown - `large-v3-turbo`: 빠른 속도 + 좋은 정확도 (VRAM ~6GB) ← **추천**
#@markdown - `medium`: 중간 (VRAM ~5GB)
#@markdown - `small`: 가벼운 모델 (VRAM ~2GB)
#@markdown - `base`: 매우 가벼운 모델 (VRAM ~1GB)
WHISPER_MODEL = "large-v3-turbo" #@param ["large-v3", "large-v3-turbo", "turbo", "large-v2", "medium", "small", "base", "tiny"]

#@markdown ### 출력 형식
OUTPUT_FORMAT = "txt" #@param ["txt", "srt"]

#@markdown ### 화자분리
ENABLE_DIARIZATION = True #@param {type:"boolean"}

#@markdown ### 채팅 수집
ENABLE_CHAT = True #@param {type:"boolean"}

#@markdown ### 화질
PREFERRED_QUALITY = "best" #@param ["best", "1080p", "720p", "480p", "360p", "worst"]

#@markdown ### 네이버 쿠키 (성인 인증용, 필요한 경우만)
#@markdown `NID_AUT=값; NID_SES=값;` 형식으로 입력
NAVER_COOKIES = "" #@param {type:"string"}

print(f"영상: {VIDEO_URL}")
print(f"구간: {START_TIME} ~ {END_TIME}")
print(f"모델: {WHISPER_MODEL}")
print(f"출력: {OUTPUT_FORMAT}")
print(f"화자분리: {'예' if ENABLE_DIARIZATION else '아니오'}")
print(f"채팅수집: {'예' if ENABLE_CHAT else '아니오'}")

## 3. 유틸리티 함수 로드

In [ ]:
import re
import os
import json
import time
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from typing import Optional, Tuple, Dict, Any, List, Union, Callable

# ==============================
# 유틸리티 함수
# ==============================

def convert_time_to_seconds(time_str: str) -> Optional[int]:
    """시간 문자열을 초로 변환"""
    try:
        parts = time_str.split(':')
        if len(parts) == 3:
            h, m, s = map(int, parts)
            return h * 3600 + m * 60 + s
        elif len(parts) == 2:
            m, s = map(int, parts)
            return m * 60 + s
        else:
            return int(parts[0])
    except ValueError:
        return None


def clean_filename(filename: str) -> str:
    """파일명에서 특수문자 제거"""
    chars_to_remove = ['\u2665', '\u2661', '\u10e6', '\u2b50', '\u3266', '\u2727', '\u300b', '\u300a',
                       '\u2660', '\u2666', '\u2764\ufe0f', '\u2663', '\u273f', '\ua34d', '\u1d17', '\u2605']
    cleaned = filename
    for char in chars_to_remove:
        cleaned = cleaned.replace(char, '')
    cleaned = re.sub(r'[/@!~*\[\]#$%^&()\-_=+<>?;:\'"\\|]', '', cleaned)
    return cleaned.strip()


def format_time(seconds: float) -> str:
    """초를 HH:MM:SS 형식으로 변환"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def generate_filename(title: str, quality: str, extension: str) -> str:
    """타임스탬프 포함 파일명 생성"""
    clean_title = clean_filename(title)
    if not clean_title:
        clean_title = "video"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{clean_title}_{quality}_{timestamp}.{extension}"


def validate_time_range(start_time: str, end_time: str) -> Tuple[Optional[int], Optional[int], Optional[str]]:
    """시간 범위 검증"""
    start_seconds = convert_time_to_seconds(start_time)
    end_seconds = convert_time_to_seconds(end_time)
    if start_seconds is None or end_seconds is None:
        return None, None, "올바른 시간 형식을 입력해주세요. (HH:MM:SS)"
    if start_seconds >= end_seconds:
        return None, None, "종료 시간이 시작 시간보다 늦어야 합니다."
    return start_seconds, end_seconds, None


def safe_file_removal(*file_paths: str) -> None:
    """파일 안전 삭제"""
    for file_path in file_paths:
        try:
            if os.path.exists(file_path):
                os.remove(file_path)
        except Exception:
            pass


print("유틸리티 함수 로드 완료")

## 4. Chzzk 다운로더 로드

In [ ]:
import ffmpeg


class ChzzkDownloader:
    """치지직(Chzzk) VOD 다운로더"""

    VOD_URL = "https://apis.naver.com/neonplayer/vodplay/v1/playback/{video_id}?key={in_key}&env=real&lc=ko&cpl=ko"
    VOD_INFO = "https://api.chzzk.naver.com/service/v3/videos/{video_no}"
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"

    @staticmethod
    def parse_cookies(cookie_string: str) -> Dict[str, str]:
        if not cookie_string:
            return {}
        cookies = {}
        try:
            cookie_string = cookie_string.strip()
            if ';' in cookie_string:
                for cookie in cookie_string.split(';'):
                    if '=' in cookie:
                        key, value = cookie.strip().split('=', 1)
                        cookies[key.strip()] = value.strip()
            else:
                for line in cookie_string.split('\n'):
                    line = line.strip()
                    if '=' in line:
                        key, value = line.split('=', 1)
                        cookies[key.strip()] = value.strip()
        except Exception:
            pass
        return cookies

    @staticmethod
    def extract_video_info(link: str) -> Tuple[Optional[str], Optional[str]]:
        patterns = [
            r'https?://chzzk\.naver\.com/video/(?P<video_no>\d+)(?:\?.*)?$',
            r'https?://chzzk\.naver\.com/(?:video/(?P<video_no>\d+)|live/(?P<channel_id>[^/?]+))(?:\?.*)?$',
            r'https?://m\.chzzk\.naver\.com/video/(?P<video_no>\d+)(?:\?.*)?$'
        ]
        for pattern in patterns:
            match = re.match(pattern, link.strip())
            if match:
                video_no = match.group("video_no")
                if video_no:
                    return video_no, None
        return None, "올바르지 않은 링크입니다. 치지직 비디오 URL을 확인해주세요."

    @staticmethod
    def get_video_streams(video_no: str, cookies=None):
        api_url = ChzzkDownloader.VOD_INFO.format(video_no=video_no)
        headers = {
            "User-Agent": ChzzkDownloader.USER_AGENT,
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
            "Referer": "https://chzzk.naver.com/",
            "Origin": "https://chzzk.naver.com"
        }
        session = requests.Session()
        if cookies:
            if isinstance(cookies, str):
                cookies = ChzzkDownloader.parse_cookies(cookies)
            session.cookies.update(cookies)

        max_retries = 3
        response = None
        for attempt in range(max_retries):
            try:
                response = session.get(api_url, headers=headers, timeout=30)
                response.raise_for_status()
                break
            except requests.RequestException as e:
                if attempt == max_retries - 1:
                    return None, f"비디오 정보를 가져오는데 실패했습니다: {str(e)}"
                time.sleep(1)

        if response.status_code == 404:
            return None, "비디오를 찾을 수 없습니다."
        elif response.status_code == 403:
            return None, "비디오에 접근할 권한이 없습니다. 쿠키를 설정해주세요."

        try:
            json_data = response.json()
            if json_data.get('code') == 403:
                return None, "성인 인증이 필요한 영상입니다. 네이버 로그인 쿠키를 입력해주세요."
            elif json_data.get('code') != 200:
                return None, f"API 오류: {json_data.get('message', '알 수 없는 오류')}"

            content = json_data.get('content', {})
            video_id = content.get('videoId')
            in_key = content.get('inKey')
            adult_status = content.get('adult', False)

            if video_id is None or in_key is None:
                if adult_status and not cookies:
                    return None, "성인 인증이 필요한 영상입니다. 네이버 로그인 쿠키를 입력해주세요."
                return None, "로그인이 필요한 비디오이거나 비공개 비디오입니다."

            video_url = ChzzkDownloader.VOD_URL.format(video_id=video_id, in_key=in_key)
            author = content.get('channel', {}).get('channelName', 'Unknown')
            title = content.get('videoTitle', 'Unknown Title')
            duration = content.get('duration', 0)
            vod_status = content.get('vodStatus', '')

            stream_qualities = None
            if vod_status == 'ABR_HLS':
                stream_qualities = ChzzkDownloader._parse_dash_manifest(video_url, cookies)
                if not stream_qualities:
                    stream_qualities = ChzzkDownloader._get_fallback_streams(video_url, cookies)
            elif vod_status == 'UPLOAD':
                live_rewind_json = content.get('liveRewindPlaybackJson')
                if live_rewind_json:
                    try:
                        playback_data = json.loads(live_rewind_json)
                        media_list = playback_data.get('media', [])
                        if media_list:
                            hls_path = media_list[0].get('path')
                            if hls_path:
                                stream_qualities = [{
                                    'quality_label': 'auto', 'resolution': 'auto',
                                    'base_url': hls_path, 'bandwidth': 0,
                                    'width': 0, 'height': 0, 'mime_type': 'application/x-mpegURL'
                                }]
                    except (ValueError, KeyError):
                        pass
                if not stream_qualities:
                    return None, "영상이 아직 인코딩 중입니다. 잠시 후 다시 시도해주세요."
            else:
                stream_qualities = ChzzkDownloader._parse_dash_manifest(video_url, cookies)
                if not stream_qualities:
                    stream_qualities = ChzzkDownloader._get_fallback_streams(video_url, cookies)

            if not stream_qualities:
                return None, "스트림 URL을 가져올 수 없습니다."

            return {
                'author': author, 'title': title, 'video_id': video_id,
                'video_no': video_no, 'duration': duration, 'adult': adult_status,
                'stream_qualities': stream_qualities
            }, None
        except Exception as e:
            return None, f"예상치 못한 오류: {str(e)}"

    @staticmethod
    def _parse_dash_manifest(video_url, cookies=None):
        headers = {
            "User-Agent": ChzzkDownloader.USER_AGENT,
            "Accept": "application/dash+xml, application/xml, text/xml, */*",
            "Referer": "https://chzzk.naver.com/"
        }
        session = requests.Session()
        if cookies:
            if isinstance(cookies, str):
                cookies = ChzzkDownloader.parse_cookies(cookies)
            session.cookies.update(cookies)

        for attempt in range(2):
            try:
                response = session.get(video_url, headers=headers, timeout=30)
                response.raise_for_status()
                content_type = response.headers.get('content-type', '').lower()
                response_text = response.text.strip()
                if 'xml' not in content_type and 'dash' not in content_type:
                    if not response_text.startswith('<?xml') and not response_text.startswith('<MPD'):
                        continue

                root = ET.fromstring(response.text)
                ns = {"mpd": "urn:mpeg:dash:schema:mpd:2011"}
                stream_qualities = []
                processed_qualities = set()

                def _process_rep(rep, fallback_url=None, fallback_mime='video/mp4'):
                    width = rep.get('width')
                    height = rep.get('height')
                    bandwidth = rep.get('bandwidth')
                    rep_id = rep.get('id', '')
                    mime = rep.get('mimeType', fallback_mime)
                    base_el = rep.find("mpd:BaseURL", namespaces=ns)
                    base_url = base_el.text if base_el is not None else fallback_url
                    if width and height and base_url:
                        if base_url.rstrip('/').endswith('/hls'):
                            return
                        qk = f"{width}x{height}_{mime}"
                        if qk not in processed_qualities:
                            processed_qualities.add(qk)
                            h = int(height)
                            label = '4K' if h >= 2160 else '1440p' if h >= 1440 else '1080p' if h >= 1080 else '720p' if h >= 720 else '480p' if h >= 480 else '360p' if h >= 360 else f'{h}p'
                            stream_qualities.append({
                                'resolution': f"{width}x{height}", 'width': int(width),
                                'height': h, 'bandwidth': int(bandwidth) if bandwidth else 0,
                                'base_url': base_url, 'id': rep_id,
                                'mime_type': mime, 'quality_label': label
                            })

                for adapt in root.findall(".//mpd:AdaptationSet", namespaces=ns):
                    mime = adapt.get('mimeType', '')
                    if 'video' in mime:
                        adapt_base = None
                        base_el = adapt.find("mpd:BaseURL", namespaces=ns)
                        if base_el is None:
                            base_el = root.find(".//mpd:BaseURL", namespaces=ns)
                        if base_el is not None:
                            adapt_base = base_el.text
                        for rep in adapt.findall("mpd:Representation", namespaces=ns):
                            _process_rep(rep, adapt_base, mime)

                if not stream_qualities:
                    root_base = None
                    rbe = root.find(".//mpd:BaseURL", namespaces=ns)
                    if rbe is not None:
                        root_base = rbe.text
                    for rep in root.findall(".//mpd:Representation", namespaces=ns):
                        _process_rep(rep, root_base)

                if stream_qualities:
                    stream_qualities.sort(key=lambda x: (x['height'], x['bandwidth']), reverse=True)
                    return stream_qualities
            except Exception:
                if attempt < 1:
                    time.sleep(0.5)
        return None

    @staticmethod
    def _get_fallback_streams(video_url, cookies=None):
        headers = {"User-Agent": ChzzkDownloader.USER_AGENT, "Accept": "*/*"}
        session = requests.Session()
        if cookies:
            if isinstance(cookies, str):
                cookies = ChzzkDownloader.parse_cookies(cookies)
            session.cookies.update(cookies)
        try:
            response = session.get(video_url, headers=headers, timeout=15)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            ns = {"mpd": "urn:mpeg:dash:schema:mpd:2011"}
            base_el = root.find(".//mpd:BaseURL", namespaces=ns)
            if base_el is not None:
                return [{'quality_label': 'auto', 'resolution': 'auto', 'base_url': base_el.text,
                         'bandwidth': 0, 'width': 0, 'height': 0, 'mime_type': 'video/mp4'}]
        except Exception:
            pass
        return None

    @staticmethod
    def get_stream_by_quality(stream_qualities, preferred_quality="best"):
        if not stream_qualities:
            return None
        if preferred_quality == "best":
            return stream_qualities[0]
        elif preferred_quality == "worst":
            return stream_qualities[-1]
        else:
            for s in stream_qualities:
                if preferred_quality.lower() in s['quality_label'].lower():
                    return s
            target = re.search(r'(\d+)', preferred_quality)
            if target:
                th = int(target.group(1))
                return min(stream_qualities, key=lambda x: abs(x['height'] - th))
        return stream_qualities[0]

    @staticmethod
    def download_video_segment(base_url, output_path, start_time, end_time,
                               progress_callback=None):
        total_duration = end_time - start_time
        if total_duration <= 0:
            return False, "잘못된 구간입니다."

        methods = [
            ChzzkDownloader._dl_method_1,
            ChzzkDownloader._dl_method_2,
            ChzzkDownloader._dl_method_3,
            ChzzkDownloader._dl_method_4
        ]
        last_error = None
        for i, method in enumerate(methods):
            try:
                success, msg = method(base_url, output_path, start_time, total_duration, progress_callback)
                if success:
                    return True, msg
                last_error = f"Method {i+1}: {msg}"
            except Exception as e:
                last_error = f"Method {i+1}: {str(e)}"
        return False, f"모든 다운로드 방법 실패. 마지막 오류: {last_error}"

    @staticmethod
    def _dl_method_1(base_url, output_path, start_time, duration, cb=None):
        process = (
            ffmpeg.input(base_url, ss=start_time, t=duration,
                         user_agent=ChzzkDownloader.USER_AGENT,
                         headers=f'Referer: https://chzzk.naver.com/\r\nUser-Agent: {ChzzkDownloader.USER_AGENT}',
                         reconnect=1, reconnect_streamed=1, reconnect_delay_max=5)
            .output(output_path, c='copy', avoid_negative_ts='make_zero',
                    fflags='+genpts', movflags='+faststart')
            .overwrite_output()
            .global_args('-progress', 'pipe:2', '-nostats', '-loglevel', 'warning')
            .run_async(pipe_stdout=True, pipe_stderr=True)
        )
        return ChzzkDownloader._monitor(process, duration, cb, output_path)

    @staticmethod
    def _dl_method_2(base_url, output_path, start_time, duration, cb=None):
        process = (
            ffmpeg.input(base_url, user_agent=ChzzkDownloader.USER_AGENT,
                         headers=f'Referer: https://chzzk.naver.com/\r\nUser-Agent: {ChzzkDownloader.USER_AGENT}',
                         reconnect=1, reconnect_streamed=1)
            .output(output_path, ss=start_time, t=duration, c='copy',
                    avoid_negative_ts='make_zero')
            .overwrite_output()
            .global_args('-progress', 'pipe:2', '-nostats', '-loglevel', 'warning')
            .run_async(pipe_stdout=True, pipe_stderr=True)
        )
        return ChzzkDownloader._monitor(process, duration, cb, output_path)

    @staticmethod
    def _dl_method_3(base_url, output_path, start_time, duration, cb=None):
        process = (
            ffmpeg.input(base_url, ss=start_time, t=duration,
                         headers=f'User-Agent: {ChzzkDownloader.USER_AGENT}')
            .output(output_path, vcodec='libx264', acodec='aac', preset='fast', crf=23)
            .overwrite_output()
            .global_args('-progress', 'pipe:2', '-nostats', '-loglevel', 'error')
            .run_async(pipe_stdout=True, pipe_stderr=True)
        )
        return ChzzkDownloader._monitor(process, duration, cb, output_path)

    @staticmethod
    def _dl_method_4(base_url, output_path, start_time, duration, cb=None):
        headers = {'User-Agent': ChzzkDownloader.USER_AGENT, 'Referer': 'https://chzzk.naver.com/'}
        response = requests.get(base_url, headers=headers, stream=True, timeout=60)
        if response.status_code != 200:
            return False, f"HTTP 오류: {response.status_code}"
        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if cb and total_size > 0:
                        cb(min((downloaded / total_size) * 100, 100))
        if not os.path.exists(output_path) or os.path.getsize(output_path) == 0:
            return False, "다운로드된 파일이 비어있습니다"
        # Extract segment
        temp_path = output_path + ".temp"
        try:
            os.rename(output_path, temp_path)
            process = (
                ffmpeg.input(temp_path, ss=start_time, t=duration)
                .output(output_path, c='copy')
                .overwrite_output()
                .global_args('-loglevel', 'error')
                .run_async(pipe_stdout=True, pipe_stderr=True)
            )
            rc = process.wait()
            try:
                os.remove(temp_path)
            except OSError:
                pass
            if rc == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                return True, "다운로드 완료 (HTTP + 세그먼트 추출)"
            return False, f"세그먼트 추출 실패 (코드: {rc})"
        except Exception as e:
            try:
                if os.path.exists(temp_path):
                    os.rename(temp_path, output_path)
            except OSError:
                pass
            return False, f"세그먼트 추출 오류: {str(e)}"

    @staticmethod
    def _monitor(process, duration, cb=None, output_path=None):
        last_progress = 0
        stderr_lines = []
        while True:
            if process.poll() is not None:
                break
            line = process.stderr.readline()
            if not line:
                time.sleep(0.1)
                continue
            decoded = line.decode('utf-8', errors='replace').strip()
            stderr_lines.append(decoded)
            if len(stderr_lines) > 20:
                stderr_lines = stderr_lines[-20:]
            if cb:
                m = re.search(r'out_time_ms=(\d+)', decoded)
                if m:
                    ct = int(m.group(1)) / 1_000_000
                    progress = min((ct / duration) * 100, 100)
                    if progress > last_progress:
                        cb(progress)
                        last_progress = progress
        rc = process.wait()
        if rc == 0 and output_path and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            return True, "다운로드 완료"
        err = '; '.join(stderr_lines[-3:]) if stderr_lines else "알 수 없는 오류"
        return False, f"다운로드 실패 (코드: {rc}): {err}"


print("Chzzk 다운로더 로드 완료")

## 5. 오디오 프로세서 로드 (음성인식 + 화자분리)

In [ ]:
import torch

# Whisper 백엔드 확인
FASTER_WHISPER_AVAILABLE = False
try:
    from faster_whisper import WhisperModel
    FASTER_WHISPER_AVAILABLE = True
    print("faster-whisper 사용 가능")
except ImportError:
    print("faster-whisper 미설치")

# 화자분리 백엔드 확인
WESPEAKER_AVAILABLE = False
SIMPLE_DIARIZER_AVAILABLE = False

try:
    import wespeaker
    WESPEAKER_AVAILABLE = True
    print("WeSpeaker 사용 가능")
except Exception:
    print("WeSpeaker 미설치")

try:
    from simple_diarizer.diarizer import Diarizer
    SIMPLE_DIARIZER_AVAILABLE = True
    print("Simple Diarizer 사용 가능")
except Exception:
    print("Simple Diarizer 미설치")


class AudioProcessor:
    """오디오 처리: 음성인식 + 화자분리"""

    def __init__(self, whisper_model="large-v3-turbo", diarization_backend="auto", use_gpu=True):
        self.whisper_model_name = whisper_model
        self.diarization_backend = diarization_backend
        self.use_gpu = use_gpu
        self.device = self._get_device()
        self.whisper = None
        self.diarization_pipeline = None

        if self.diarization_backend == "auto":
            if WESPEAKER_AVAILABLE:
                self.diarization_backend = "wespeaker"
            elif SIMPLE_DIARIZER_AVAILABLE:
                self.diarization_backend = "simple"
            else:
                self.diarization_backend = "none"

        print(f"디바이스: {self.device}")
        print(f"Whisper 모델: {self.whisper_model_name}")
        print(f"화자분리: {self.diarization_backend}")

    def _get_device(self):
        if not self.use_gpu:
            return "cpu"
        if torch.cuda.is_available():
            return "cuda"
        return "cpu"

    def load_models(self):
        """Whisper 및 화자분리 모델 로드"""
        if self.whisper is None and FASTER_WHISPER_AVAILABLE:
            print(f"Whisper 모델 로딩 중... ({self.whisper_model_name}, {self.device})")
            compute_type = "float16" if self.device == "cuda" else "int8"
            self.whisper = WhisperModel(
                self.whisper_model_name,
                device=self.device,
                compute_type=compute_type
            )
            print("Whisper 모델 로드 완료")

        if self.diarization_pipeline is None:
            self._load_diarization_model()

    def _load_diarization_model(self):
        if self.diarization_backend == "wespeaker":
            try:
                print("WeSpeaker 모델 로딩 중...")
                model = wespeaker.load_model('english')
                if self.device == "cuda":
                    model.set_device('cuda:0')
                self.diarization_pipeline = ("wespeaker", model)
                print("WeSpeaker 모델 로드 완료")
            except Exception as e:
                print(f"WeSpeaker 로드 실패: {e}")
        elif self.diarization_backend == "simple":
            try:
                print("Simple Diarizer 모델 로딩 중...")
                diar = Diarizer(embed_model='ecapa', cluster_method='sc')
                self.diarization_pipeline = ("simple", diar)
                print("Simple Diarizer 모델 로드 완료")
            except Exception as e:
                print(f"Simple Diarizer 로드 실패: {e}")

    def extract_audio(self, video_path, audio_path):
        """비디오에서 오디오 추출"""
        try:
            (
                ffmpeg.input(video_path)
                .output(audio_path, acodec='pcm_s16le', ac=1, ar='16000')
                .overwrite_output()
                .run(quiet=True)
            )
            return True, "오디오 추출 완료"
        except Exception as e:
            return False, f"오디오 추출 실패: {str(e)}"

    def perform_diarization(self, audio_path, num_speakers=None):
        """화자분리 수행"""
        if self.diarization_pipeline is None:
            return None
        backend_type, model = self.diarization_pipeline
        try:
            if backend_type == "wespeaker":
                print("WeSpeaker 화자분리 진행 중...")
                result = model.diarize(audio_path, 'utterance')
                if not result:
                    return None
                segments = []
                for item in result:
                    if len(item) >= 3:
                        segments.append({'start': float(item[0]), 'end': float(item[1]), 'speaker': str(item[2])})
                speakers = set(s['speaker'] for s in segments)
                print(f"화자분리 완료 - 감지된 화자 수: {len(speakers)}")
                return segments
            elif backend_type == "simple":
                print("Simple Diarizer 화자분리 진행 중...")
                kwargs = {}
                if num_speakers:
                    kwargs['num_speakers'] = num_speakers
                result = model.diarize(audio_path, **kwargs)
                segments = []
                for seg in result:
                    segments.append({'start': float(seg['start']), 'end': float(seg['end']),
                                    'speaker': str(seg.get('label', 'UNKNOWN'))})
                speakers = set(s['speaker'] for s in segments)
                print(f"화자분리 완료 - 감지된 화자 수: {len(speakers)}")
                return segments
        except Exception as e:
            print(f"화자분리 실패: {str(e)}")
            return None

    def transcribe(self, audio_path):
        """음성인식 수행"""
        if not FASTER_WHISPER_AVAILABLE or self.whisper is None:
            return None, "Whisper 모델이 로드되지 않았습니다."
        try:
            print("음성인식 진행 중...")
            segments_gen, info = self.whisper.transcribe(
                audio_path, language="ko", beam_size=5, vad_filter=True
            )
            segments = []
            full_text_parts = []
            for seg in segments_gen:
                segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text})
                full_text_parts.append(seg.text.strip())
            result = {'segments': segments, 'text': ' '.join(full_text_parts), 'language': info.language}
            print(f"음성인식 완료 - {len(segments)}개 세그먼트")
            return result, None
        except Exception as e:
            return None, f"음성인식 실패: {str(e)}"

    def create_transcript(self, whisper_result, diarization=None):
        """트랜스크립트 생성 (TXT 형식)"""
        segments = whisper_result['segments']
        if diarization is None:
            lines = []
            for seg in segments:
                lines.append(f"[{format_time(seg['start'])} - {format_time(seg['end'])}] {seg['text'].strip()}")
            return "\n".join(lines)

        speaker_map = {}
        next_id = 1
        lines = []
        for seg in segments:
            raw_speaker = self._find_speaker(diarization, seg['start'], seg['end'])
            if raw_speaker not in speaker_map:
                speaker_map[raw_speaker] = f"화자{next_id}"
                next_id += 1
            speaker = speaker_map[raw_speaker]
            lines.append(f"[{format_time(seg['start'])} - {format_time(seg['end'])}] {speaker}: {seg['text'].strip()}")
        print(f"화자 매핑: {len(speaker_map)}명 ({', '.join(speaker_map.values())})")
        return "\n".join(lines)

    def create_srt_transcript(self, whisper_result, diarization=None):
        """SRT 자막 생성"""
        segments = whisper_result['segments']
        srt = []
        speaker_map = {}
        next_id = 1
        for i, seg in enumerate(segments, 1):
            text = seg['text'].strip()
            if diarization:
                raw_speaker = self._find_speaker(diarization, seg['start'], seg['end'])
                if raw_speaker not in speaker_map:
                    speaker_map[raw_speaker] = f"화자{next_id}"
                    next_id += 1
                text = f"{speaker_map[raw_speaker]}: {text}"
            start_srt = self._to_srt_time(seg['start'])
            end_srt = self._to_srt_time(seg['end'])
            srt.append(f"{i}")
            srt.append(f"{start_srt} --> {end_srt}")
            srt.append(text)
            srt.append("")
        return "\n".join(srt)

    def _find_speaker(self, diarization, start, end):
        best_overlap = 0
        best_speaker = "UNKNOWN"
        for seg in diarization:
            overlap = max(0, min(seg['end'], end) - max(seg['start'], start))
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = seg['speaker']
        return best_speaker

    def _to_srt_time(self, seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        ms = int((seconds % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


print("\n오디오 프로세서 로드 완료")

## 6. 채팅 수집기 로드

In [ ]:
class ChatCollector:
    """치지직 채팅 수집기"""

    USER_AGENT = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    @staticmethod
    def ms_to_timestamp(ms):
        seconds = ms // 1000
        h = seconds // 3600
        m = (seconds % 3600) // 60
        s = seconds % 60
        return f"[{h:02d}:{m:02d}:{s:02d}]"

    @staticmethod
    def clean_cookies(cookies_input):
        if not cookies_input or not cookies_input.strip():
            return None
        cleaned = re.sub(r'\s+', ' ', cookies_input.replace('\n', ' ').replace('\r', ' ')).strip()
        if not cleaned.endswith(';'):
            cleaned += ';'
        return cleaned

    @staticmethod
    def extract_chat_message(chat):
        try:
            profile = json.loads(chat.get("profile", "{}"))
            nickname = profile.get("nickname", "Unknown")
            content = chat.get("content", "")
            player_time = chat.get("playerMessageTime", 0)
            ts = ChatCollector.ms_to_timestamp(player_time)
            prefix = "[도네이션] " if chat.get("messageTypeCode") == 10 else ""
            return f"{ts} {prefix}[{nickname}] : {content}"
        except Exception as e:
            return f"[ERROR] 채팅 파싱 실패: {str(e)}"

    @staticmethod
    def collect(video_no, auth_cookies=None, start_time_ms=None, end_time_ms=None):
        """채팅 수집"""
        headers = {
            "User-Agent": ChatCollector.USER_AGENT,
            "Referer": f"https://chzzk.naver.com/video/{video_no}",
        }
        if auth_cookies:
            cleaned = ChatCollector.clean_cookies(auth_cookies)
            if cleaned:
                headers["Cookie"] = cleaned

        base_url = f"https://api.chzzk.naver.com/service/v1/videos/{video_no}/chats"

        all_chats = []
        current_time = start_time_ms if start_time_ms is not None else 0
        request_count = 0
        max_requests = 10000

        while request_count < max_requests:
            params = {"playerMessageTime": current_time, "previousVideoChatSize": 50}
            try:
                response = requests.get(base_url, headers=headers, params=params, timeout=15)
            except requests.exceptions.RequestException as e:
                print(f"네트워크 오류 (req {request_count}): {e}")
                time.sleep(2)
                continue

            if response.status_code != 200:
                print(f"API 요청 실패: HTTP {response.status_code}")
                if response.status_code == 403:
                    print("403: 인증 쿠키가 필요할 수 있습니다.")
                break

            try:
                data = response.json()
            except (ValueError, KeyError):
                print("JSON 파싱 실패")
                break

            if data.get("code") != 200:
                print(f"API 오류: {data.get('message', 'Unknown')}")
                break

            content = data.get("content", {})
            batch = content.get("previousVideoChats", []) + content.get("videoChats", [])
            if not batch:
                break

            for chat in batch:
                pt = chat.get("playerMessageTime", 0)
                if start_time_ms is not None and pt < start_time_ms:
                    continue
                if end_time_ms is not None and pt > end_time_ms:
                    continue
                msg = ChatCollector.extract_chat_message(chat)
                all_chats.append((pt, msg))

            next_time = content.get("nextPlayerMessageTime")
            if next_time is None or next_time <= current_time:
                break
            if end_time_ms is not None and next_time > end_time_ms:
                break

            current_time = next_time
            request_count += 1

            if request_count % 100 == 0:
                elapsed_s = current_time // 1000
                h, m, s = elapsed_s // 3600, (elapsed_s % 3600) // 60, elapsed_s % 60
                print(f"  {request_count} requests, {len(all_chats)} chats, at {h:02d}:{m:02d}:{s:02d}")

            time.sleep(0.3)

        unique = sorted(set(all_chats), key=lambda x: x[0])
        messages = [m for _, m in unique]
        print(f"채팅 수집 완료: {len(messages)}개 메시지 ({request_count} requests)")
        return messages


print("채팅 수집기 로드 완료")

## 7. 실행 - 영상 정보 확인
영상 정보를 가져오고 사용 가능한 화질을 확인합니다.

In [ ]:
# 시간 검증
start_seconds, end_seconds, time_error = validate_time_range(START_TIME, END_TIME)
if time_error:
    raise ValueError(f"시간 오류: {time_error}")

print(f"구간: {START_TIME} ({start_seconds}초) ~ {END_TIME} ({end_seconds}초)")
print(f"총 길이: {end_seconds - start_seconds}초\n")

# 영상 정보 가져오기
video_no, error = ChzzkDownloader.extract_video_info(VIDEO_URL)
if error:
    raise ValueError(f"URL 오류: {error}")

print(f"비디오 번호: {video_no}")

cookies = NAVER_COOKIES if NAVER_COOKIES.strip() else None
stream_data, error = ChzzkDownloader.get_video_streams(video_no, cookies)
if error:
    raise ValueError(f"스트림 오류: {error}")

print(f"\n=== 영상 정보 ===")
print(f"제목: {stream_data['title']}")
print(f"채널: {stream_data['author']}")
print(f"길이: {stream_data['duration']}초")
if stream_data.get('adult'):
    print("⚠️ 성인 인증 영상")

print(f"\n=== 사용 가능한 화질 ===")
for i, q in enumerate(stream_data['stream_qualities']):
    if q['quality_label'] == 'auto':
        print(f"  {i+1}. auto")
    else:
        print(f"  {i+1}. {q['quality_label']} ({q['resolution']}) - {q['bandwidth']:,} bps")

# 화질 선택
selected_stream = ChzzkDownloader.get_stream_by_quality(
    stream_data['stream_qualities'], PREFERRED_QUALITY
)
print(f"\n선택된 화질: {selected_stream['quality_label']}")

## 8. 실행 - 다운로드 & 트랜스크립트 생성
비디오 다운로드, 오디오 추출, 음성인식, 화자분리를 순차적으로 수행합니다.

In [ ]:
from IPython.display import display, HTML, clear_output

# 작업 디렉토리 준비
WORK_DIR = "/content/chzzk_output"
os.makedirs(WORK_DIR, exist_ok=True)

quality_suffix = selected_stream['quality_label'] if selected_stream['quality_label'] != 'auto' else 'auto'
video_path = os.path.join(WORK_DIR, generate_filename(stream_data['title'], quality_suffix, 'mp4'))
audio_path = os.path.join(WORK_DIR, generate_filename(stream_data['title'], quality_suffix, 'wav'))
transcript_path = os.path.join(WORK_DIR, generate_filename(stream_data['title'], quality_suffix, OUTPUT_FORMAT))

# ========================================
# Step 1: 채팅 수집 (비디오 다운로드와 병렬 처리 불가능하므로 먼저)
# ========================================
chat_messages = []
chat_path = None

if ENABLE_CHAT:
    print("=" * 50)
    print("Step 1/5: 채팅 수집 중...")
    print("=" * 50)

    start_ms = start_seconds * 1000
    end_ms = end_seconds * 1000
    chat_messages = ChatCollector.collect(video_no, cookies, start_ms, end_ms)

    if chat_messages:
        chat_filename = generate_filename(stream_data['title'], quality_suffix, 'txt')
        chat_filename = chat_filename.replace('.txt', '_chat.txt')
        chat_path = os.path.join(WORK_DIR, chat_filename)
        with open(chat_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(chat_messages))
        print(f"채팅 저장 완료: {chat_path}")
    else:
        print("채팅이 없거나 수집할 수 없습니다.")
else:
    print("Step 1/5: 채팅 수집 건너뜀")

# ========================================
# Step 2: 비디오 다운로드
# ========================================
print(f"\n{'=' * 50}")
print("Step 2/5: 비디오 다운로드 중...")
print(f"{'=' * 50}")

last_reported = [0]
def progress_cb(progress):
    pct = int(progress)
    if pct >= last_reported[0] + 10:
        print(f"  다운로드: {pct}%")
        last_reported[0] = pct

success, msg = ChzzkDownloader.download_video_segment(
    selected_stream['base_url'], video_path, start_seconds, end_seconds, progress_cb
)
if not success:
    raise RuntimeError(f"다운로드 실패: {msg}")
print(f"다운로드 완료: {video_path}")
print(f"파일 크기: {os.path.getsize(video_path) / 1024 / 1024:.1f} MB")

# ========================================
# Step 3: 오디오 추출
# ========================================
print(f"\n{'=' * 50}")
print("Step 3/5: 오디오 추출 중...")
print(f"{'=' * 50}")

processor = AudioProcessor(
    whisper_model=WHISPER_MODEL,
    diarization_backend="auto" if ENABLE_DIARIZATION else "none",
    use_gpu=True
)

success, msg = processor.extract_audio(video_path, audio_path)
if not success:
    raise RuntimeError(f"오디오 추출 실패: {msg}")
print(f"오디오 추출 완료: {audio_path}")

# ========================================
# Step 4: 모델 로드 + 음성인식 + 화자분리
# ========================================
print(f"\n{'=' * 50}")
print("Step 4/5: AI 모델 로드 및 음성인식 중...")
print(f"{'=' * 50}")

processor.load_models()

# 화자분리
diarization = None
if ENABLE_DIARIZATION:
    print("\n화자분리 수행 중...")
    diarization = processor.perform_diarization(audio_path)

# 음성인식
print("\n음성인식 수행 중...")
whisper_result, error = processor.transcribe(audio_path)
if error:
    raise RuntimeError(f"음성인식 실패: {error}")

# ========================================
# Step 5: 트랜스크립트 생성
# ========================================
print(f"\n{'=' * 50}")
print("Step 5/5: 트랜스크립트 생성 중...")
print(f"{'=' * 50}")

if OUTPUT_FORMAT == "srt":
    transcript = processor.create_srt_transcript(whisper_result, diarization)
else:
    transcript = processor.create_transcript(whisper_result, diarization)

with open(transcript_path, 'w', encoding='utf-8') as f:
    f.write(transcript)

print(f"트랜스크립트 저장 완료: {transcript_path}")

# 임시 파일 정리
safe_file_removal(video_path, audio_path)

print(f"\n{'=' * 50}")
print("모든 처리 완료!")
print(f"{'=' * 50}")

## 9. 결과 확인

In [ ]:
from IPython.display import display, HTML

# 트랜스크립트 표시
print("=" * 60)
print(f"트랜스크립트 ({stream_data['title']})")
print(f"구간: {START_TIME} ~ {END_TIME}")
print("=" * 60)
print(transcript)

# 채팅 표시
if chat_messages:
    print(f"\n{'=' * 60}")
    print(f"채팅 로그 ({len(chat_messages)}개)")
    print("=" * 60)
    for msg in chat_messages[:50]:  # 처음 50개만 표시
        print(msg)
    if len(chat_messages) > 50:
        print(f"\n... 외 {len(chat_messages) - 50}개 메시지 (전체 내용은 파일 다운로드)")

## 10. 파일 다운로드
생성된 파일을 로컬 PC로 다운로드합니다.

In [ ]:
try:
    from google.colab import files

    # 트랜스크립트 다운로드
    if os.path.exists(transcript_path):
        print(f"트랜스크립트 다운로드: {os.path.basename(transcript_path)}")
        files.download(transcript_path)

    # 채팅 다운로드
    if chat_path and os.path.exists(chat_path):
        print(f"채팅 로그 다운로드: {os.path.basename(chat_path)}")
        files.download(chat_path)

    print("\n다운로드 완료!")

except ImportError:
    # Colab이 아닌 환경
    print("Google Colab 환경이 아닙니다. 파일 경로를 확인하세요:")
    print(f"  트랜스크립트: {transcript_path}")
    if chat_path:
        print(f"  채팅 로그: {chat_path}")